In [1]:
print("heello world")

heello world


## Post Generation Autonomus workflow with Iterative Approach - HUMAN IN THE LOOP

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from typing import Literal,List
from dotenv import load_dotenv
import os

load_dotenv()  # Load environment variables from .env file

True

In [ ]:
model = ChatGoogleGenerativeAI(api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-2.5-flash-lite", temperature=0)


In [11]:
class GenerationState(BaseModel):
    topic: str = Field(..., description="The main topic for content generation")
    post : str = Field(..., description="The generated blog post content")
    evaluated_post: Literal["approved", "not_approved"] = Field(..., description="Evaluation of the generated post")
    feedback: str = Field(..., description="Feedback for improving the post if not approved")

    feedback_history: List[str] = Field(..., description="History of feedback provided")
    post_history: List[str] = Field(..., description="History of generated posts")
    iteration: int = Field(0, description="Current iteration number")
    max_iterations: int = Field(3, description="Maximum number of iterations allowed")
    

In [ ]:
# functions for each node state
def generate_post(state: GenerationState) -> dict:

    messages = [
        SystemMessage(content="You are a creative blog post generator."),
        HumanMessage(content=f"Generate a detailed blog post on the topic: {state.topic}.")
    ]

    post = model.invoke(messages).content
    return {"post": post}

def evaluate_post(state: GenerationState) -> dict:
    prompt = f"""Evaluate the following blog post and determine if it is approved or not approved. Provide feedback if not approved.
    
    Blog Post:
    {state.post}
    
    Respond with 'approved' or 'not_approved' and provide feedback if necessary."""



In [ ]:
graph = StateGraph(GenerationState)

# add modes
graph.add_node('generate_post',generate_post)
graph.add_node('evaluate_post',evaluate_post)
graph.add_node('optimize_post',optimize_post)


# add edges
graph.add_edge(START, 'generate_post')
graph.add_edge('generate_post', 'evaluate_post')
